In [1]:
import os,sys 

PROJECT_ROOT = "/home/qing/raid/paperwork/kgtool"
sys.path.append(PROJECT_ROOT)



import pickle
from qdls.data import save_json,load_json
from loguru import logger   
from src_smolagents.tools import tools 

from smolagents.agents import ToolCallingAgent
from smolagents.models import LiteLLMModel

2025-02-13 17:23:03.807 | INFO     | src_smolagents.tools:load_or_create_engine:24 - Loaded KB from pickle file


In [2]:
model = LiteLLMModel(
    model_id="gpt-4o-mini", # This model is a bit weak for agentic behaviours though
    api_base=os.environ["CHATANY_API_URL"] , 
    api_key=  os.environ["CHATANY_API_KEY"] , 
    num_ctx=8192 
)

TOOL_CALLING_EXAMPLES = """
Question: Seymour Cassel got nominated for Academy Award for best supporting actor, tell me the subject of this statement.
Thought: First, I need to understand who Seymour Cassel is. I will search for him.
Action: {"tool": "Find", "args": {"name": "Seymour Cassel"}}
Observation: Output: (['Q708153'], None)\n Seymour Cassel is found, note as $0.
Thought: Now I need to understand what the "Academy Award for Best Supporting Actor" refers to. I will search for that.
Action: {"tool": "Find", "args": {"name": "Academy Award for best supporting actor"}}
Observation: Output:(['Q106291'], None)\n Academy Award for best supporting actor is found, note as $1.
Thought: Next, I will query the relation attribute of the relation "nominated for" between "Seymour Cassel" and "Academy Award for best supporting actor".
Action: {"tool": "QueryRelationQualifier", "args": {"s_entities": $0, "t_entities": $1, "relation": "nominated for", "qkey": "statement is subject of"}}
Observation: Output: '41st Academy Awards'\nThe subject of the statement is 41st Academy Awards.
Final Answer: 41st Academy Awards

Question: Does My Neighbor Totoro or Hannah Arendt, originally in German, possess the longer run-time?
Thought:I need to compare the run-time lengths of two films: My Neighbor Totoro and Hannah Arendt (in German) to determine which one is longer.  I will find entity "My Neighbor Totoro" first.
Action: {"tool": "Find", "args": {"name": "My Neighbor Totoro"}}
Observation: Output: (['Q39571'], None)\n My Neighbor Totoro is found, note as $0.
Thought: The second file "Hannah Arendt" is restricted to be a German film. Next, I will look up "German" to search for information on films originally in the German language.
Action: {"tool": "Find", "args": {"name": "German"}}
Observation: Output: (['Q188'], None)\n German is found, note as $1.
Thought: Now, I need to relate the German language to films that have German as their original language.
Action: {"tool": "Relate", "args": {"entities": $1, "relation": "original language of film or TV show", "direction": "backward"}}
Observation: Output: (['Q150804', ...], [{'object': 'Q150804', 'direction':'backward', 'qualifiers': [], 'relation': 'original language of film or TV show'}, ...])\n All German films are returned, note as $2.
Thought: Then I need to locate the entity "Hannah Arendt" so that I can get the desired entity by intersecting the two sets. 
Action: {"tool": "Find", "args": {"name": "Hannah Arendt"}}
Observation: Output: (['Q60025', 'Q1575773'], None)\n There are two films named Hannah Arendt, note as $3.
Thought: A simple intersection of the two sets can result the film originally in German.
Action: {"tool": "And", "args": {"entities": $2, "entities": $3}}
Observation: Output: (['Q1575773'], None)\n The film in the question is filtered out.
Thought: Now I can compare the duration of My Neighbor Totoro and the German-language Hannah Arendt. 
Action: {"tool": "SelectBetween", "args": {"l_entities": $0, "r_entities": $4, "key": "duration", "op": "greater"}}
Observation: Output: 'Hannah Arendt'\n The answer is found.
Final Answer: Hannah Arendt

""" 
from smolagents.prompts import 
agent = ToolCallingAgent(
    tools=tools,
    model=model,
    max_steps=10,
    system_prompt=TOOL_CALLING_EXAMPLES,
)

In [3]:
question= "Which town has a TOID of 4000000074573917 and has an OS grid reference of SP8778?"
agent.run(question) 

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which town has a TOID of 4000000074573917 and has an OS grid reference of SP8778?                               │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'FilterStr' with arguments: {'entities': '()', 'key': 'TOID', 'value': '4000000074573917'}        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Error whene executing tool FilterStr with arguments {'entities': '()', 'key': 'TOID', 'value': '4000000074573917'}:
ValueError: not enough values to unpack (expected 2, got 0)
You should only use this tool with a correct input.
As a reminder, this tool's description is the following: 'Filter entities based on string attribute key-value 
pairs'.
It takes inputs: {'entities': {'type': 'string', 'description': 'A tuple, first element is entity list, second is 
triple list'}, 'key': {'type': 'string', 'description': 'Attribute key'}, 'value': {'type': 'string', 
'description': 'Attribute value'}} and returns output type string

[Step 0: Duration 3.60 seconds| Input tokens: 5,073 | Output tokens: 68]

...

Error whene executing tool FilterStr with arguments {'entities': "(['Q7270', 'Q130232', 'Q280658', 'Q8355', 
'Q8445', 'Q1640319', 'Q5043', 'Q289', 'Q7168625', 'Q912985', 'Q14212', 'Q11404', 'Q188473', 'Q6607', 'Q163829', 
'Q44148', 'Q21198', 'Q381243', 'Q33999', 'Q7889', 'Q2405480', 'Q1860', 'Q40831', 'Q42178', 'Q219557', 'Q7163', 
'Q9794', 'Q11023', 'Q11366', 'Q5994', 'Q7174', 'Q736559', 'Q30461', 'Q36180', 'Q36279', 'Q11424', 'Q7749', 
'Q16533', 'Q484641', 'Q901', 'Q852767', 'Q432', 'Q8341', 'Q52954', 'Q28389', 'Q1371925', 'Q19842222', 'Q49451', 
'Q484876', 'Q37073', 'Q46185', 'Q17884', 'Q2617520', 'Q810447', 'Q319221', 'Q182015', 'Q639669', 'Q2095173', 
'Q1344', 'Q132050', 'Q132311', 'Q483501', 'Q23540', 'Q1136507', 'Q43845', 'Q4610556', 'Q24925', 'Q93204', 
'Q1054574', 'Q9135', 'Q7614320', 'Q902104', 'Q7325', 'Q11399', 'Q395', 'Q1734', 'Q3142', 'Q164444', 'Q645928', 
'Q1327500', 'Q12202', 'Q11425', 'Q183816', 'Q4115013', 'Q1142885', 'Q189004', 'Q482980', 'Q747802', 'Q173799', 
'Q170238', 'Q11268', 'Q615150', 'Q850412', 'Q2920921', 'Q36834', 'Q116', 'Q2066131', 'Q269770', 'Q46970', 
'Q206159', 'Q753110', 'Q855091', 'Q7317272', 'Q309', 'Q2095', 'Q947873', 'Q179805', 'Q349', 'Q1259759', 'Q83440', 
'Q638', 'Q167037', 'Q38848', 'Q45981', 'Q35032', 'Q340169', 'Q9798', 'Q423208', 'Q76239', 'Q15078788', 'Q41254', 
'Q170435', 'Q8343', 'Q1048902', 'Q9759', 'Q180744', 'Q5891', 'Q389735', 'Q128758', 'Q420', 'Q42989', 'Q8434', 
'Q1044427', 'Q177275', 'Q18123741', 'Q34749', 'Q11401', 'Q7748', 'Q5', 'Q685884', 'Q955824', 'Q875538', 'Q373342', 
'Q43343', 'Q8242', 'Q83364', 'Q181247', 'Q862086', 'Q39809', 'Q41614', 'Q1298934', 'Q1196408', 'Q39338', 'Q40348', 
'Q42240', 'Q187760', 'Q26718', 'Q735', 'Q19020', 'Q334453', 'Q3071', 'Q11635', 'Q186472', 'Q506240', 'Q83460', 
'Q213714', 'Q1589009', 'Q20378', 'Q170790', 'Q727928', 'Q170611', 'Q851213', 'Q37226', 'Q131272', 'Q11190', 
'Q1080794', 'Q35874', 'Q20502', 'Q211573', 'Q1107', 'Q7188', 'Q80993', 'Q25372', 'Q58339', 'Q3918', 'Q179164', 
'Q102932', 'Q658255', 'Q8253', 'Q265662', 'Q5372', 'Q11707', 'Q34057', 'Q173453', 'Q932586', 'Q193355', 'Q192431', 
'Q1798603', 'Q9778', 'Q9237', 'Q170082', 'Q1758037', 'Q684740', 'Q128124', 'Q180268', 'Q131524', 'Q184485', 
'Q82955', 'Q7850', 'Q8261', 'Q2743', 'Q11032', 'Q19860', 'Q276453', 'Q483251', 'Q49892', 'Q10922', 'Q377910', 
'Q1143635', 'Q822146', 'Q191489', 'Q1075651', 'Q39631', 'Q486415', 'Q10737', 'Q23442', 'Q217199', 'Q336', 
'Q2973201', 'Q487965', 'Q47783', 'Q230262', 'Q1930187', 'Q36747', 'Q188784', 'Q183862', 'Q202866', 'Q184644', 
'Q12206', 'Q6452410', 'Q4830453', 'Q149086', 'Q218616', 'Q105420', 'Q93184', 'Q837837', 'Q211723', 'Q80930', 
'Q483352', 'Q144', 'Q486839', 'Q24862', 'Q11461', 'Q189290', 'Q203720', 'Q105513', 'Q1369558', 'Q727781', 
'Q5322166', 'Q42201', 'Q1323698', 'Q482', 'Q1703577', 'Q1211934', 'Q186424', 'Q47566', 'Q2855609', 'Q4184', 
'Q9174', 'Q192096', 'Q272167', 'Q33506', 'Q581714', 'Q216337', 'Q41583', 'Q9301', 'Q719927', 'Q8361', 'Q200779', 
'Q220505', 'Q8458', 'Q185867', 'Q38926', 'Q36192', 'Q472055', 'Q7397', 'Q83307', 'Q3286546', 'Q726', 'Q83267', 
'Q1515757', 'Q1542343', 'Q1377182', 'Q6430706', 'Q837171', 'Q1650915', 'Q846662', 'Q1920219', 'Q11016', 'Q226730', 
'Q388906', 'Q308876', 'Q131436', 'Q159334', 'Q275596', 'Q428602', 'Q23653', 'Q2418495', 'Q486365', 'Q936942', 
'Q729', 'Q270948', 'Q523', 'Q781132', 'Q8465', 'Q333156', 'Q211302', 'Q101849', 'Q203184', 'Q191600', 'Q165', 
'Q7310', 'Q61509', 'Q943995', 'Q14656', 'Q16560', 'Q4175034', 'Q2658935', 'Q39594', 'Q12135', 'Q744038', 'Q160738',
'Q31207', 'Q1165777', 'Q7696995', 'Q16970', 'Q37901', 'Q9826', 'Q4282636', 'Q599151', 'Q11417', 'Q9134', 'Q33881', 
'Q4', 'Q12298619', 'Q215536', 'Q210932', 'Q12570', 'Q193076', 'Q593644', 'Q81938', 'Q178197', 'Q1358344', 'Q80006',
'Q146', 'Q46169', 'Q2977', 'Q4692', 'Q846224', 'Q223335', 'Q163740', 'Q8436', 'Q1261214', 'Q1336920', 'Q185029', 
'Q43229', 'Q21010853', 'Q889821', 'Q215380', 'Q12178928', 'Q476028', 'Q5398426', 'Q181

[Step 2: Duration 99.11 seconds| Input tokens: 109,198 | Output tokens: 4,343]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 3: Duration 8.38 seconds| Input tokens: 207,969 | Output tokens: 8,610]

KeyboardInterrupt: 

In [4]:
agent.memory.steps

[TaskStep(task='Which town has a TOID of 4000000074573917 and has an OS grid reference of SP8778?', task_images=None),
 ActionStep(model_input_messages=None, tool_calls=None, start_time=1739438177.5942762, end_time=1739438177.598424, step_number=1, error=None, duration=0.004147768020629883, model_output_message=None, model_output=None, observations=None, observations_images=None, action_output=None),
 ActionStep(model_input_messages=None, tool_calls=None, start_time=1739438177.5997138, end_time=1739438177.6012557, step_number=2, error=None, duration=0.0015418529510498047, model_output_message=None, model_output=None, observations=None, observations_images=None, action_output=None),
 ActionStep(model_input_messages=None, tool_calls=None, start_time=1739438177.6023078, end_time=1739438177.6035159, step_number=3, error=None, duration=0.0012080669403076172, model_output_message=None, model_output=None, observations=None, observations_images=None, action_output=None),
 ActionStep(model_inpu

In [6]:
agent.system_prompt